In [1]:
import numpy as np
import matplotlib.pyplot as plt 
import pandas as pd
import os
import re
import torch
import torch.nn as nn
import math
from torch.utils.data import Dataset, DataLoader

In [2]:
index_df = pd.read_csv(
    r"D:/DACN/tf-test/tf_modification_ms2/data/_index.tsv",
    header=None
)

print(index_df.head())

             0
0  1/17705.tsv
1  1/27289.tsv
2  1/26084.tsv
3   1/7517.tsv
4  1/26511.tsv


In [3]:
base_dir = r"D:/DACN/tf-test/tf_modification_ms2/data/251128"

dfs = []

for i, rel_path in enumerate(index_df[0]):
    full_path = os.path.join(base_dir, rel_path)
    
    temp = pd.read_csv(full_path, sep="\t")
    temp["psm_id"] = f"psm_{i}"  
    
    dfs.append(temp)



In [4]:
df = pd.concat(dfs, ignore_index=True)

print(df.head())
print(f"Total rows: {len(df)}")

   PrecursorCharge            ModifiedPeptide FragmentLossType  \
0                2  _AEEDEILNRS(UniMod:21)PR_           noloss   
1                2  _AEEDEILNRS(UniMod:21)PR_           noloss   
2                2  _AEEDEILNRS(UniMod:21)PR_           noloss   
3                2  _AEEDEILNRS(UniMod:21)PR_           noloss   
4                2  _AEEDEILNRS(UniMod:21)PR_           noloss   

   FragmentNumber FragmentType FragmentCharge  RelativeIntensity psm_id  
0               1            y              1           0.085714  psm_0  
1               2            y              1           1.000000  psm_0  
2               3            y              1           0.028571  psm_0  
3               4            y              1           0.014286  psm_0  
4               5            y              1           0.057143  psm_0  
Total rows: 35702


In [27]:
frag_types = [
        "b_z1", "b_z2", "y_z1", "y_z2",
        "b_modloss_z1", "b_modloss_z2",
        "y_modloss_z1", "y_modloss_z2"
    ]

In [28]:
import pandas as pd
import numpy as np
import re

def fast_preprocess(df):

    df['clean_seq'] = df['ModifiedPeptide'].str.strip('_').str.replace(r'\(.*?\)', '', regex=True)
    df['L'] = df['clean_seq'].str.len()
    df['charge_num'] = df['FragmentCharge'].astype(str).str.extract('(\d+)').astype(int)
    df['loss_clean'] = df['FragmentLossType'].apply(lambda x: '' if x == 'noloss' else '_modloss')
    df['target_col'] = df['FragmentType'] + df['loss_clean'] + '_z' + df['charge_num'].astype(str)
    
    
    pivot_df = df.pivot_table(
        index=['psm_id', 'FragmentNumber'], 
        columns='target_col', 
        values='RelativeIntensity', 
        aggfunc='max' 
    )

    
    pivot_df = pivot_df.reindex(columns=frag_types, fill_value=0.0).fillna(0)

    
    results = {}
    for psm_id, group in pivot_df.groupby(level=0):
       
        L = df[df['psm_id'] == psm_id]['L'].iloc[0]
        
     
        actual_matrix = group.loc[(psm_id, 1):(psm_id, L-1)]
        
       
        if len(actual_matrix) < (L - 1):
            actual_matrix = actual_matrix.reindex(
                pd.MultiIndex.from_product([[psm_id], range(1, L)]), 
                fill_value=0.0
            )
            
        results[psm_id] = actual_matrix.values

    return results


results = fast_preprocess(df)

# Kiểm tra thử psm_0
if 'psm_0' in results:
    print(f"--- Kết quả cho psm_0 ---")
    print(f"Shape của ma trận: {results['psm_0'].shape}") 

--- Kết quả cho psm_0 ---
Shape của ma trận: (11, 8)


In [49]:
sample_id = 'psm_170'
group = df[df['psm_id'] == sample_id]
L = group['L'].iloc[29]

debug_matrix = pd.DataFrame(0.0, index=range(1, L), columns=frag_types)

for _, row in group.iterrows():
    target = row['target_col']
    frag = int(row['FragmentNumber'])
    val = row['RelativeIntensity']
    
    if target in frag_types:
        debug_matrix.at[frag, target] = max(
            debug_matrix.at[frag, target], val
        )

print(f"--- Kiểm tra Peptide: {group['ModifiedPeptide'].iloc[0]} (L={L}) ---")
print(debug_matrix)


--- Kiểm tra Peptide: _AVTIANS(UniMod:21)PSKPSEKDSVVSLESQK_ (L=24) ---
     b_z1   b_z2  y_z1   y_z2  b_modloss_z1  b_modloss_z2  y_modloss_z1  \
1   0.000  0.000  0.00  0.000         0.000           0.0         0.000   
2   0.000  0.000  0.02  0.000         0.000           0.0         0.020   
3   0.200  0.000  0.12  0.000         0.140           0.0         0.000   
4   0.060  0.000  0.06  0.000         0.140           0.0         0.012   
5   0.040  0.000  0.06  0.000         0.040           0.0         0.000   
6   0.020  0.000  0.20  0.000         0.008           0.0         0.008   
7   0.000  0.000  0.40  0.000         0.080           0.0         0.040   
8   0.000  0.000  0.08  0.000         0.000           0.0         0.000   
9   0.000  0.000  0.20  0.000         0.020           0.0         0.000   
10  0.080  0.000  0.20  0.000         0.060           0.0         0.012   
11  0.000  0.000  0.10  0.000         0.000           0.0         0.000   
12  0.000  0.000  0.04  0.000

In [48]:
# Khởi tạo dictionary để lưu kết quả mỗi PSM
psm_nonzero_z2 = {}

# Lặp qua từng PSM
for sample_id, group in df.groupby('psm_id'):
    L = group['L'].iloc[0]
    
    # Khởi tạo matrix fragment
    debug_matrix = pd.DataFrame(0.0, index=range(1, L), columns=frag_types)
    
    # Fill matrix
    for _, row in group.iterrows():
        target = row['target_col']
        frag = int(row['FragmentNumber'])
        val = row['RelativeIntensity']
        
        if target in frag_types:
            debug_matrix.at[frag, target] = max(debug_matrix.at[frag, target], val)
    
    # Chỉ giữ fragment b/z 2 > 0
    b_y_z2_cols = ['b_z2', 'y_z2', 'b_modloss_z2', 'y_modloss_z2']
    nonzero_fragments = debug_matrix[b_y_z2_cols][debug_matrix[b_y_z2_cols] > 0].dropna(how='all')
    
    # Nếu có ít nhất 1 fragment >0, lưu lại
    if not nonzero_fragments.empty:
        psm_nonzero_z2[sample_id] = {
            'peptide': group['ModifiedPeptide'].iloc[0],
            'matrix': nonzero_fragments
        }

# --- In ra kết quả ---
for psm_id, info in psm_nonzero_z2.items():
    print(f"PSM ID: {psm_id}, Peptide: {info['peptide']}")
    print(info['matrix'], "\n")

PSM ID: psm_170, Peptide: _AVTIANS(UniMod:21)PSKPSEKDSVVSLESQK_
     b_z2   y_z2  b_modloss_z2  y_modloss_z2
13    NaN  0.012           NaN           NaN
14  0.008    NaN           NaN           NaN
16  0.014    NaN           NaN           NaN
17  0.016    NaN           NaN           NaN 

PSM ID: psm_171, Peptide: _AVTIANS(UniMod:21)PSKPSEKDSVVSLESQK_
        b_z2  y_z2  b_modloss_z2  y_modloss_z2
16  0.016667   NaN           NaN           NaN
18       NaN  0.05           NaN           NaN
19  0.010000   NaN           NaN           NaN 

PSM ID: psm_172, Peptide: _AVTIANS(UniMod:21)PSKPSEKDSVVSLESQK_
     b_z2  y_z2  b_modloss_z2  y_modloss_z2
13    NaN  0.01           NaN           NaN
16  0.010   NaN           NaN           NaN
19  0.009   NaN           NaN           NaN 

PSM ID: psm_173, Peptide: _AVTIANS(UniMod:21)PSKPSEKDSVVSLESQK_
      b_z2  y_z2  b_modloss_z2  y_modloss_z2
13     NaN  0.01           NaN           NaN
16  0.0150   NaN           NaN           NaN
17  0.0125   N

In [51]:
# Tìm độ dài lớn nhất thực tế trong chuỗi AA sạch
clean_seqs = df['ModifiedPeptide'].str.replace(r'\(UniMod:\d+\)', '', regex=True).str.strip('_')
actual_max = clean_seqs.str.len().max() + 2 # +2 cho N và C term

print(f"Peptide dài nhất là: {actual_max}. Chốt max_len = {actual_max}")

# Khởi tạo dataset với con số thực tế đó


Peptide dài nhất là: 40. Chốt max_len = 40


In [30]:
df["target_col"].unique()

array(['y_z1', 'y_modloss_z1', 'b_z1', 'b_modloss_z1', 'y_z2', 'b_z2'],
      dtype=object)

In [31]:
df.head()

,PrecursorCharge,ModifiedPeptide,FragmentLossType,FragmentNumber,FragmentType,FragmentCharge,RelativeIntensity,psm_id,target_col,clean_seq,L,charge_num,loss_clean
0,2,_AEEDEILNRS(UniMod:21)PR_,noloss,1,y,1,0.085714,psm_0,y_z1,AEEDEILNRSPR,12,1,
1,2,_AEEDEILNRS(UniMod:21)PR_,noloss,2,y,1,1.000000,psm_0,y_z1,AEEDEILNRSPR,12,1,
2,2,_AEEDEILNRS(UniMod:21)PR_,noloss,3,y,1,0.028571,psm_0,y_z1,AEEDEILNRSPR,12,1,
3,2,_AEEDEILNRS(UniMod:21)PR_,noloss,4,y,1,0.014286,psm_0,y_z1,AEEDEILNRSPR,12,1,
4,2,_AEEDEILNRS(UniMod:21)PR_,noloss,5,y,1,0.057143,psm_0,y_z1,AEEDEILNRSPR,12,1,


In [33]:
import re
import pandas as pd

def parse_peptdeep_format(mod_peptide):
    # 1. Loại bỏ gạch dưới ở đầu/cuối
    peptide = mod_peptide.strip('_')
    
    # 2. Tìm tất cả các cụm (Modification) và vị trí của chúng
    # Dùng regex để tìm nội dung trong ngoặc đơn
    pattern = re.compile(r'([A-Z])(\([^)]+\))')
    
    clean_seq = re.sub(r'\([^)]+\)', '', peptide) # Chuỗi chỉ toàn Amino Acid
    nAA = len(clean_seq)
    
    mods = []
    mod_sites = []
    
    # Tìm các modification trên residue (A-Z)
    matches = list(re.finditer(r'([A-Z])(\([^)]+\))', peptide))
    
    # Tính toán vị trí thực tế trên chuỗi đã làm sạch
    current_clean_idx = 0
    temp_peptide = peptide
    
    # Cách đơn giản hơn để lấy site: 
    # Duyệt qua từng phần tách bởi regex
    parts = re.split(r'(\([^)]+\))', peptide)
    curr_site = 0
    for p in parts:
        if p.startswith('('):
            mods.append(p[1:-1]) # Bỏ dấu ngoặc
            mod_sites.append(curr_site) # Gán vào vị trí AA ngay trước đó
        else:
            curr_site += len(p)
            
    return clean_seq, ";".join(mods), ";".join([str(s) for s in mod_sites]), nAA

In [35]:
# Giả sử df_unique là bảng chứa các psm_id duy nhất
df_meta = df.drop_duplicates('psm_id').copy()

# Áp dụng hàm tách
parsed_data = df_meta['ModifiedPeptide'].apply(parse_peptdeep_format)
df_meta[['sequence', 'mods', 'mod_sites', 'nAA']] = pd.DataFrame(parsed_data.tolist(), index=df_meta.index)

# Bây giờ df_meta đã có đủ 4 cột: sequence, mods, mod_sites, nAA

In [36]:
df_meta.head()

,PrecursorCharge,ModifiedPeptide,FragmentLossType,FragmentNumber,FragmentType,FragmentCharge,RelativeIntensity,psm_id,target_col,clean_seq,L,charge_num,loss_clean,sequence,mods,mod_sites,nAA
0,2,_AEEDEILNRS(UniMod:21)PR_,noloss,1,y,1,0.085714,psm_0,y_z1,AEEDEILNRSPR,12,1,,AEEDEILNRSPR,UniMod:21,10,12
64,2,_AEEDEILNRS(UniMod:21)PR_,noloss,1,y,1,0.100000,psm_1,y_z1,AEEDEILNRSPR,12,1,,AEEDEILNRSPR,UniMod:21,10,12
128,2,_AEEDEILNRS(UniMod:21)PR_,noloss,1,y,1,0.083333,psm_2,y_z1,AEEDEILNRSPR,12,1,,AEEDEILNRSPR,UniMod:21,10,12
188,2,_AEEDEILNRS(UniMod:21)PR_,noloss,1,y,1,0.100000,psm_3,y_z1,AEEDEILNRSPR,12,1,,AEEDEILNRSPR,UniMod:21,10,12
249,2,_AEEDEILNRS(UniMod:21)PR_,noloss,1,y,1,0.087500,psm_4,y_z1,AEEDEILNRSPR,12,1,,AEEDEILNRSPR,UniMod:21,10,12


In [42]:
import sys
import os

sys.path.append(r'd:\DACN\tf-test\tf_modification_ms2')

In [45]:
df["ModifiedPeptide"].unique()

array(['_AEEDEILNRS(UniMod:21)PR_', '_AGDLLEDS(UniMod:21)PKRPK_',
       '_AGLGS(UniMod:21)PERPPK_',
       '_AHT(UniMod:21)PTPGIYMGRPTHSGGGGGGGGGGGGGGGGR_',
       '_AKS(UniMod:21)PQPPVEEEDEHFDDTVVC(UniMod:4)LDTYNC(UniMod:4)DLHFK_',
       '_ALASEKS(UniMod:21)PTADAKPAPK_', '_ALSAS(UniMod:21)HTDLAH_',
       '_ANGQENGHVKS(UniMod:21)NGDLSPK_',
       '_ANS(UniMod:21)PEKPPEAGAAHKPR_', '_ASAVSELS(UniMod:21)PR_',
       '_ASGQAFELILS(UniMod:21)PR_',
       '_(UniMod:1)AS(UniMod:21)GVAVSDGVIK_',
       '_(UniMod:1)AS(UniMod:21)GVQVADEVC(UniMod:4)R_',
       '_AS(UniMod:21)PAPGSGHPEGPGAHLDMNSLDR_',
       '_ATAPQTQHVS(UniMod:21)PMR_', '_ATNES(UniMod:21)EDEIPQLVPIGK_',
       '_ATS(UniMod:21)NVFAMFDQSQIQEFK_',
       '_AVTIANS(UniMod:21)PSKPSEKDSVVSLESQK_',
       '_AVT(UniMod:21)PVPTKTEEVSNLK_',
       '_C(UniMod:4)GSGPVHISGQHLVAVEEDAES(UniMod:21)EDEEEEDVK_',
       '_C(UniMod:4)GSGPVHISGQHLVAVEEDAES(UniMod:21)EDEEEEDVKLLSISGK_',
       '_DELHIVEAEAMNYEGS(UniMod:21)PIKVTLATLK_',
       '_DGQ

In [43]:



from model.featurize import get_batch_aa_indices, get_batch_mod_feature, parse_instrument_indices


aa_indices = get_batch_aa_indices(df_meta['sequence'].values)

# 2. Lấy Modification Features (Ma trận hóa học 109 chiều)
# Kết quả: (Batch_size, nAA + 2, 109)
mod_x = get_batch_mod_feature(df_meta)

# 3. Lấy Instrument Indices
# Kết quả: List các con số (0, 1, 2...)
instrument_indices = parse_instrument_indices(["QE"] * len(df_meta))

ModuleNotFoundError: No module named 'alphabase.modification'